In [13]:
import pandas as pd

In [14]:
ciclos = [
    ("2009", "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/BMX_F.XPT",
             "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/DEMO_F.XPT"),
    ("2011", "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/BMX_G.XPT",
             "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DEMO_G.XPT"),
    ("2013", "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BMX_H.XPT",
             "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/DEMO_H.XPT"),
    ("2015", "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BMX_I.XPT",
             "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DEMO_I.XPT"),
    ("2017", "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.XPT",
             "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.XPT"),
]

In [18]:

dfs = []
for year, bmx_url, demo_url in ciclos:
    bmx  = pd.read_sas(bmx_url,  encoding='utf-8')[['SEQN', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST']]
    demo_raw = pd.read_sas(demo_url, encoding='utf-8')
    
    # RIDRETH3 solo existe desde 2011
    demo_cols = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1']
    if 'RIDRETH3' in demo_raw.columns:
        demo_cols.append('RIDRETH3')
    demo = demo_raw[demo_cols]

    merged = pd.merge(bmx, demo, on='SEQN')
    merged['Ciclo'] = year
    dfs.append(merged)

df = pd.concat(dfs, ignore_index=True)

In [19]:
df = df.rename(columns={
    'SEQN':     'ID',
    'BMXWT':    'Peso',
    'BMXHT':    'Altura',
    'BMXBMI':   'IMC',
    'BMXWAIST': 'Circunferencia_Cintura',
    'RIAGENDR': 'Genero',
    'RIDAGEYR': 'Edad',
    'RIDRETH1': 'Raza_V1',    # Compatible con 1999-2010, Asian agrupado en "5"
    'RIDRETH3': 'Raza_V2',    # Desde 2011, Asian separado como "6"
})

In [20]:
df.to_csv('NHANES_2009_2018.csv', index=False)
print(df.shape)
print(df.head())

(47652, 10)
        ID   Peso  Altura    IMC  Circunferencia_Cintura  Genero  Edad  \
0  51624.0   87.4   164.7  32.22                   100.4     1.0  34.0   
1  51625.0   17.0   105.4  15.30                    49.0     1.0   4.0   
2  51626.0   72.3   181.3  22.00                    74.7     1.0  16.0   
3  51627.0   39.8   147.8  18.22                    63.0     1.0  10.0   
4  51628.0  116.8   166.0  42.39                   118.2     2.0  60.0   

   Raza_V1 Ciclo  Raza_V2  
0      3.0  2009      NaN  
1      5.0  2009      NaN  
2      4.0  2009      NaN  
3      4.0  2009      NaN  
4      4.0  2009      NaN  


In [21]:
# DICCIONARIOS DE VARIABLES

dic_genero = {
    1: 'Hombre',
    2: 'Mujer'
}

dic_raza_v1 = {
    1: 'Mexicano-Americano',
    2: 'Otro Hispano',
    3: 'Blanco No Hispano',
    4: 'Negro No Hispano',
    5: 'Otra Raza / Multirracial No Hispano'
}

dic_raza_v2 = {
    1: 'Mexicano-Americano',
    2: 'Otro Hispano',
    3: 'Blanco No Hispano',
    4: 'Negro No Hispano',
    6: 'Asiático No Hispano',
    7: 'Otra Raza / Multirracial No Hispano'
}